<div style="background-color:#B5121B;color:#ffffff;padding:16px 22px;border-radius:6px;font-family:Arial, Helvetica, sans-serif;">
<div style="font-size:12px;letter-spacing:0.08em;text-transform:uppercase;opacity:0.85;">PHYS7101 &middot; Numerical Methods and Data Analysis</div>
<div style="font-size:21px;font-weight:700;margin-top:6px;line-height:1.3;">Week 3 — Derivatives and Integrals from Samples</div>
</div>

#### Finite differences and the truncation/round-off trade-off, the CO₂ growth rate, the trapezoidal and Simpson's rules, and why grids fail in many dimensions.
***

## Where this fits

Interpolation builds a curve from samples. The same idea gives derivatives and integrals: if you only have $f$ at discrete points (or can only evaluate it), you estimate slopes and areas from those points.

Two threads from Week 1 come back:

- **round-off:** a finite difference subtracts two nearly equal numbers, the catastrophic cancellation from `week1_foundations.ipynb`;
- **RK4:** for $dy/dt = f(t)$, an RK4 step *is* Simpson's rule.

Lecture 4 covers the methods. Today you will:

1. write forward and central differences, find the step size that minimises the error, and improve on it with Richardson extrapolation;
2. differentiate the CO₂ record to get its growth rate;
3. see how differentiating measured values amplifies their noise, and how to fight it;
4. write the trapezoidal and Simpson's rules and measure how fast they converge;
5. use them on a real physics integral (the large-amplitude pendulum), where the "better" rule loses.

**Nothing here is assessed.** Write your predictions in your logbook before you run each comparison.

## How to use this notebook today

**Core (about 1¼ hours):** Sections 1, 2, 3, 4 and 6(a). Parts marked *Optional* (the second derivative, Sections 5 and 7, 6(b) and the extensions) can wait.

**Working on Exercise 1?** Sections 1, 3 and 4 give you the tools for Tasks 4 and 5: central differences, the noise they pick up, and Simpson's rule. Section 6 is the same elliptic integral as Task 4.3. Do those, then use the rest of the lab for the exercise.

**Behind on earlier weeks?** Catch up first on what Exercise 1 builds on: `week1_euler_rk4_lab.ipynb` (Sections 1–3 and 5), `week2_root_finding.ipynb` (Sections 1, 2 and 5), `week2_fitting.ipynb` (Sections 2–4) and `week2_code_and_plots.ipynb` (Section 6). This notebook will still be here.

In [ ]:
import os

# Data: read from a local datasets/ folder if you've cloned/downloaded the
# repository, otherwise straight from GitHub (e.g. in Colab). Same file either way.
DATA_URL = "https://raw.githubusercontent.com/MatSmithAstro/phys7101_2026_resources/main/workshops/week3/datasets/"

def data_path(filename):
    local = os.path.join("datasets", filename)
    return local if os.path.exists(local) else DATA_URL + filename

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

## <span style="color:#B5121B">1. Finite differences</span>

From the Taylor series, $f(x + h) = f(x) + h f'(x) + \tfrac{h^2}{2} f''(x) + \tfrac{h^3}{6} f'''(x) + \dots$, so

$$\text{forward:}\quad f'(x) \approx \frac{f(x + h) - f(x)}{h}, \qquad \text{error} \approx \tfrac{h}{2} f''(x),$$

$$\text{central:}\quad f'(x) \approx \frac{f(x + h) - f(x - h)}{2h}, \qquad \text{error} \approx \tfrac{h^2}{6} f'''(x).$$

This is the *truncation* error: it shrinks as $h \to 0$. But the numerator subtracts two nearly equal numbers, each with a relative round-off error of about $\epsilon \approx 2\times10^{-16}$ (Week 1). That gives a *round-off* error of about $\epsilon |f| / h$, which grows as $h \to 0$.

**(a)** Write `forward_diff(f, x, h)` and `central_diff(f, x, h)`.

In [ ]:
def forward_diff(f, x, h):
    """Forward-difference estimate of f'(x)."""
    raise NotImplementedError

def central_diff(f, x, h):
    """Central-difference estimate of f'(x)."""
    raise NotImplementedError

In [ ]:
try:
    print(forward_diff(np.sin, 1.0, 1e-3), central_diff(np.sin, 1.0, 1e-3), "exact", np.cos(1.0))
except NotImplementedError:
    print("Not implemented yet: complete the function above, then re-run this cell.")

**(b)** **Predict first**, in your logbook: what value of $h$ gives the most accurate derivative? Is smaller always better?

For $f(x) = \sin x$ at $x = 1$, compute the absolute error of both methods for $h$ from $10^{-16}$ to $1$ (`np.logspace(-16, 0, 161)`). Plot both on log–log axes. Add the lines $h/2$ and $h^2/6$ (truncation) and $\epsilon/h$ (round-off) for comparison.

In [ ]:
hs = np.logspace(-16, 0, 161)
eps = np.finfo(float).eps

# TODO: errors of forward_diff and central_diff for each h, compared with np.cos(1.0)
# TODO: log-log plot with the truncation and round-off guide lines
# TODO: print the h that gives the smallest error for each method, and that error

In your logbook: the best possible accuracy of each method, and the step size that achieves it. Why does central win twice over (better accuracy *and* a larger, safer step)?

**(c) Richardson extrapolation.** Lecture 4 showed that if the error is $Ch^p$ with known $p$, two estimates $A(h)$ and $A(h/2)$ combine to cancel the leading term:

$$A \approx \frac{2^p A(h/2) - A(h)}{2^p - 1}.$$

For the central difference $p = 2$. At $x = 1$, compute the central difference and the Richardson estimate for $h = 0.1$, $0.01$ and $0.001$. How do the errors compare with the best plain central difference from (b)?

In [ ]:
# TODO: central_diff at h and h/2; Richardson combination; errors against np.cos(1.0)

## <span style="color:#FD9029">Optional: the second derivative</span>

$f''(x) \approx [f(x+h) - 2f(x) + f(x-h)]/h^2$, with truncation error $\tfrac{h^2}{12}f^{(4)}(x)$. Repeat (b) for $f''(1)$ of $\sin x$ (exact: $-\sin 1$). Where is the best $h$ now, and why has it moved?

In [ ]:
# Optional

## <span style="color:#B5121B">2. Differentiating real data: how fast is CO₂ rising?</span>

With data you can't choose $h$: it's the sampling interval (one month here), and the noise is far larger than round-off.

**(a)** Load the CO₂ record (as in `week3_interpolation.ipynb`) and estimate $dy/dt$ in ppm per year with `np.gradient(y, t)`, which uses central differences (and handles uneven spacing). Plot it. Is it the growth rate you were hoping to see?

In [ ]:
co2 = pd.read_csv(data_path("co2_mm_mlo.csv"), comment="#")
t = co2["decimal date"].to_numpy()
y = co2["average"].to_numpy()

In [ ]:
# TODO: rate = np.gradient(y, t); plot it against t; print its range

**(b)** A 12-month central difference, $[y(t + 6\,\text{mo}) - y(t - 6\,\text{mo})]$ per year, compares the same point of the seasonal cycle on both sides, so the cycle cancels. Compute it (`(y[12:] - y[:-12]) / (t[12:] - t[:-12])`, centred at `(t[12:] + t[:-12]) / 2`). Plot it and print the mean growth rate for each decade. What has happened to the growth rate since the 1960s?

In [ ]:
# TODO: 12-month difference; plot; decade means

In your logbook: why does the 12-month difference remove the seasonal cycle? What would a 6-month difference do?

## <span style="color:#B5121B">3. Differentiating noisy data</span>

With data you can't choose $h$, and the values are not exact. Lecture 4 (Part C) showed what happens: differentiating measured values amplifies their noise. Here is the lecture's example: $x(t) = \cos t$ sampled every $\Delta t$ on $[0, 2\pi)$, each sample with independent noise of standard deviation $\sigma = 0.01$. The exact velocity is $-\sin t$, so you can measure the error.

**(a)** Compute the central-difference velocity at the interior samples, $v_i = (x_{i+1} - x_{i-1})/(2\Delta t)$, using array slicing rather than a loop, and its rms error against $-\sin t$. Compare it with $\sigma/(\sqrt{2}\,\Delta t)$, the noise expected for independent errors. In your logbook: where does the $\sqrt{2}$ come from? How big is the truncation error, at most $\Delta t^2/6$ here, by comparison?

In [ ]:
rng = np.random.default_rng(4)
sigma = 0.01

def noisy_cos(dt):
    """Samples of cos(t) on [0, 2 pi) every dt, with independent Gaussian noise of standard deviation sigma."""
    t = np.arange(0, 2 * np.pi, dt)
    return t, np.cos(t) + sigma * rng.standard_normal(len(t))

dt = 0.049
t_n, x_n = noisy_cos(dt)
print(f"{len(t_n)} samples, dt = {dt}")

In [ ]:
# TODO: v = central difference at the interior points (use the slices x_n[2:] and x_n[:-2])
# TODO: rms error against -np.sin(t_n[1:-1]); compare with sigma / (np.sqrt(2) * dt)

**(b)** **Predict first:** if you sample 5 times faster ($\Delta t / 5$), does the velocity get better or worse? Repeat (a) and check against $\sigma/(\sqrt{2}\,\Delta t)$.

In [ ]:
# TODO: t5, x5 = noisy_cos(dt / 5); repeat (a)

**(c)** Lecture 4: *fit locally, then differentiate.* `scipy.signal.savgol_filter(x_n, window, 3, deriv=1, delta=dt)` fits a cubic by least squares to each `window` neighbouring samples and returns its slope. Compute its rms error for windows of 11, 21, 41 and 81 samples, leaving out half a window at each end. Which window is best, and what goes wrong when the window is too wide?

In [ ]:
from scipy.signal import savgol_filter

# TODO: for window in [11, 21, 41, 81]: v_s = savgol_filter(...); k = window // 2
#       rms error over the points k ... len - k, against -sin(t)

## <span style="color:#B5121B">4. Numerical integration</span>

Split $[a, b]$ into $n$ intervals of width $h = (b - a)/n$, with points $x_0, \dots, x_n$.

**Trapezoidal rule:** join neighbouring points with straight lines,

$$\int_a^b f\,dx \approx h\left[\tfrac12 f(x_0) + f(x_1) + \dots + f(x_{n-1}) + \tfrac12 f(x_n)\right], \qquad \text{error} \propto h^2.$$

**Simpson's rule** ($n$ even): fit a parabola through each pair of intervals,

$$\int_a^b f\,dx \approx \frac{h}{3}\left[f(x_0) + 4f(x_1) + 2f(x_2) + 4f(x_3) + \dots + 4f(x_{n-1}) + f(x_n)\right], \qquad \text{error} \propto h^4.$$

**(a)** Write `trapezoid(f, a, b, n)` and `simpson(f, a, b, n)`. Use NumPy slicing rather than a loop (`fx[1:-1:2]` picks the odd-indexed points). Raise a `ValueError` in `simpson` if `n` is odd.

In [ ]:
def trapezoid(f, a, b, n):
    """Trapezoidal-rule estimate of the integral of f from a to b with n intervals."""
    raise NotImplementedError

def simpson(f, a, b, n):
    """Simpson's-rule estimate of the integral of f from a to b with n (even) intervals."""
    raise NotImplementedError

In [ ]:
try:
    print(trapezoid(np.sin, 0, np.pi, 10), simpson(np.sin, 0, np.pi, 10), "exact 2")
except NotImplementedError:
    print("Not implemented yet: complete the function above, then re-run this cell.")

**(b)** **Predict first:** if you double $n$, by what factor should each error fall?

For $\int_0^\pi \sin x\,dx = 2$, compute both errors for $n = 4, 8, 16, \dots, 1024$. Plot error against $h$ on log–log axes and measure the slopes (`np.polyfit` on the logs). Then check your answers against `scipy.integrate.trapezoid`, `scipy.integrate.simpson` and `scipy.integrate.quad`.

In [ ]:
ns = 2 ** np.arange(2, 11)

# TODO: errors for each n; log-log plot against h = pi / n; fitted slopes
# TODO: compare with scipy.integrate

## <span style="color:#FD9029">5. Optional: RK4 is Simpson's rule</span>

In Week 1 you wrote RK4 for $dy/dt = f(t, y)$. If $f$ depends only on $t$, solving $dy/dt = f(t)$ from $y(a) = 0$ to $t = b$ *is* computing $\int_a^b f\,dt$.

With $f = f(t)$ only, the RK4 stages are $k_1 = f(t)$, $k_2 = k_3 = f(t + h/2)$, $k_4 = f(t + h)$, so one step is

$$y_{n+1} = y_n + \frac{h}{6}\left[f(t) + 4 f(t + h/2) + f(t + h)\right],$$

which is Simpson's rule on $[t, t + h]$ (with Simpson's spacing $h/2$). Lecture 4 gives the others: Euler becomes the left-rectangle rule and RK2 the midpoint rule, so the orders you measured in Week 1 are the orders of these integration rules.

Check it: integrate $dy/dt = \sin t$ from 0 to π with RK4 using $N$ steps, and compare with `simpson(np.sin, 0, np.pi, 2 * N)`.

In [ ]:
def rk4_step(f, t, y, h):
    """One RK4 step for dy/dt = f(t, y): the same stages as your Week 1 RK4 loop."""
    k1 = f(t, y)
    k2 = f(t + h / 2, y + h / 2 * k1)
    k3 = f(t + h / 2, y + h / 2 * k2)
    k4 = f(t + h, y + h * k3)
    return y + h / 6 * (k1 + 2 * k2 + 2 * k3 + k4)

In [ ]:
# TODO: for N in [2, 5, 10, 50]: step RK4 from t = 0 to pi with dy/dt = sin(t), y(0) = 0
#       compare with simpson(np.sin, 0, np.pi, 2 * N)

## <span style="color:#B5121B">6. A real integral: the large-amplitude pendulum</span>

The small-angle period of a pendulum is $T_0 = 2\pi\sqrt{L/g}$. At amplitude $\theta_0$ the exact period is

$$\frac{T}{T_0} = \frac{2}{\pi}\int_0^{\pi/2} \frac{d\phi}{\sqrt{1 - k^2 \sin^2\phi}}, \qquad k = \sin(\theta_0/2).$$

The integral is the complete elliptic integral $K$, which SciPy provides as `scipy.special.ellipk(m)` with $m = k^2$ (note: $k^2$, not $k$), so you have an exact answer to check against.

**Predict first:** which rule will be more accurate with $n = 16$, trapezoid or Simpson?

For $\theta_0 = 10°, 90°, 150°, 170°, 179°$, compute $T/T_0$ with both rules ($n = 16$), and print the exact value and both errors.

In [ ]:
from scipy.special import ellipk

# TODO: for each amplitude: k = sin(theta0 / 2); integrand = lambda phi: 1 / sqrt(1 - k^2 sin^2 phi)
#       T/T0 from trapezoid and simpson with n = 16; exact = 2 / pi * ellipk(k**2)

**(b) Optional.** For $\theta_0 = 179°$: how large must $n$ be for the trapezoid to reach $10^{-6}$? What does `scipy.integrate.quad` give, and how many function evaluations does it use (`full_output=1`, then `info["neval"]`)?

In [ ]:
# TODO

## <span style="color:#FD9029">7. Optional: why integration gets harder in higher dimensions</span>

On a grid with $n$ points per dimension, a $d$-dimensional integral needs $n^d$ function evaluations. Suppose you can afford $N = 10^6$.

Integrate $f(\mathbf{x}) = e^{-|\mathbf{x}|^2}$ over the cube $[-1, 1]^d$, for which the exact answer is $\left[\sqrt{\pi}\,\text{erf}(1)\right]^d$. The function below applies the trapezoidal rule on a grid with $n$ points per dimension.

**Predict first:** with a fixed budget of $10^6$ evaluations, how does the error change from $d = 1$ to $d = 6$?

For each $d$ from 1 to 6, set $n = \text{round}(N^{1/d})$, compute the relative error and the time taken. Then work out how many evaluations $d = 10$ would need with $n = 10$, and how long that would take at the rate you measured.

In [ ]:
from scipy.special import erf

def trapezoid_nd(d, n):
    """Trapezoidal rule for exp(-|x|^2) over [-1, 1]^d with n points per dimension."""
    x = np.linspace(-1, 1, n)
    w = np.full(n, 2 / (n - 1))
    w[[0, -1]] /= 2
    grids = np.meshgrid(*([x] * d), indexing="ij", sparse=True)
    weights = np.meshgrid(*([w] * d), indexing="ij", sparse=True)
    r2 = sum(g**2 for g in grids)
    W = 1.0
    for wk in weights:
        W = W * wk            # broadcasts to the full n^d grid of weights
    return np.sum(W * np.exp(-r2))

exact_1d = np.sqrt(np.pi) * erf(1)

In [ ]:
import time
N = 10**6

# TODO: for d in 1..6: n = round(N ** (1/d)); time trapezoid_nd(d, n); relative error vs exact_1d ** d
# TODO: estimate the cost of d = 10 with n = 10

In your logbook: how does the error scale with $N$ for the trapezoidal rule in $d$ dimensions? Would Simpson's rule ($h^4$) rescue this?

## <span style="color:#FD9029">Extension A (optional): random points instead of a grid</span>

Instead of a grid, evaluate $f$ at $N$ **random** points uniform in $[-1, 1]^d$, and estimate the integral as (volume) × (mean of $f$). Its error falls as $N^{-1/2}$ *whatever the dimension*. Use `np.random.default_rng(1)` and $N = 10^6$ for $d = 1, 3, 6, 10$, and compare the relative errors with Section 7. Where does the random method win? (This is Monte Carlo integration.)

In [ ]:
# TODO

## <span style="color:#FD9029">Extension B (optional): how fast does the Lune rise?</span>

Back to the Lancaster Quay tidal record from Weeks 1 and 2 (`lancaster_quay_tidal.csv`, also in this week's `datasets/` folder). Readings are every 15 minutes, with one 30-minute gap.

1. Convert the time to hours since the first reading. Interpolate the level onto a uniform 15-minute grid with `np.interp` (Section 1 of `week3_interpolation.ipynb`, used for real).
2. Compute $dh/dt$ (m/hour) with `np.gradient`, and find the fastest rise and the fastest fall.
3. Lecture 4: *fit locally, then differentiate*. `scipy.signal.savgol_filter(level, window_length, 3, deriv=1, delta=0.25)` fits a cubic to each window of neighbouring points by least squares and returns the slope of that cubic. Try windows of 5, 9, 13 and 17 points (1 to 4 hours). How do the fastest rise and fall change?
4. Plot a day of level and rate together. Does the water rise and fall at the same speed?

In [ ]:
# TODO

## Summary

- **Finite differences:** central ($h^2$) beats forward ($h$). The best step balances truncation against round-off, near $\epsilon^{1/3}$ for central differences. Smaller is not always better.
- **Richardson extrapolation** in $h$ removes the leading error when its order is known.
- **Differentiating data** amplifies fast variations and noise: independent errors $\sigma$ give $\sigma/(\sqrt2\,\Delta t)$ in a central difference, so sampling faster makes it worse. Choose the difference, or fit locally first, to suit the signal you want.
- **Quadrature:** trapezoid $h^2$, Simpson $h^4$, but the integrand matters: for smooth periodic integrands the trapezoid can be far better. For hard integrands use an adaptive method (`quad`).
- **RK4 for $dy/dt = f(t)$ is Simpson's rule.**
- **Dimensionality:** grid methods need $n^d$ points, so their accuracy for a fixed budget collapses as $d$ grows.

In your logbook: one result from today that surprised you, and why.

<hr style="border:none;border-top:1px solid #BEC0C2;margin-top:32px;">
<div style="color:#555656;font-size:12px;margin-top:8px;font-family:Arial, Helvetica, sans-serif;">
PHYS7101 &mdash; Numerical Methods and Data Analysis
</div>